# 05 - Customer Segmentation

Goal: group customers by behavior (Recency, Frequency, Monetary) using KMeans.

We do NOT assume K=4 up front. We test a range of K values using the elbow
method and silhouette score, inspect the actual resulting clusters, and only
then decide — and only then assign business labels, based on what the numbers
actually show.

**Cross-version compatibility fix:** this environment's pandas (3.0) defaults to the new `StringDtype` for text columns (`future.infer_string=True`). Anything fitted or computed here (OneHotEncoder categories, product-name lookups) would then get pickled with that dtype -- which older/different pandas installs (e.g. on the machine actually running the dashboard) cannot unpickle, raising `NotImplementedError` on load. Forcing `pd.set_option('future.infer_string', False)` up front keeps everything on plain `object` dtype, which pickles compatibly across pandas versions.


**Fix applied to the labeling heuristic:** the original logic assigned "New / Low Engagement" to whichever remaining cluster had the *lower* mean Recency. Those two clusters sit 99.9 vs 102.9 days apart -- noise, not signal -- while Frequency differs sharply (5.9 vs 1.8 orders), the trait that actually distinguishes them. Re-keyed that split on Frequency. Segment sizes: New=1,494, Regular=1,862, Lapsed=1,646, Best=876.


We cluster on `Recency`, `Frequency_log`, `Monetary_log` — the log-transformed
versions from notebook 04. Raw Frequency/Monetary would let a handful of
wholesale-scale customers dominate the distance calculation (see notebook 04
for why: Monetary spans £2.95 to £580,987).

## Choosing K: elbow method and silhouette score

**Silhouette score is actually highest at K=2 (0.42), not K=4.** This is worth
being honest about rather than ignoring: statistically, the single cleanest split
in this data is just "high-engagement / low-engagement" customers. K=3 and K=4
have lower silhouette scores but aren't dramatically worse (0.40 and 0.36), and
the elbow curve doesn't show one obvious "true" elbow either — it's a gradual
bend, typical for RFM data with no sharp natural clusters.

So this isn't a case where the math hands us an unambiguous answer. We inspect
the actual cluster contents at a few K values below before deciding, because
a segmentation that's statistically "purest" but business-useless (just
"good customers / bad customers") isn't actually more useful than a slightly
less clean split that separates distinct, actionable groups.

## Inspecting cluster contents at K=2, 3, 4

At K=2: just "active/valuable" vs "inactive/low-value" — true, but too coarse
to act on (a marketing team can't do much with two buckets).

At K=4, four genuinely distinct groups emerge:
- Recent, low frequency/spend — likely new or one-time customers
- Very high recency (i.e. long since last purchase), low frequency/spend — lapsed/lost
- Very recent, high frequency, high spend — best customers
- Moderate recency, moderate frequency/spend — regulars

K=3 collapses the "new" and "lapsed" groups into one "low value" bucket, which
loses a distinction that actually matters for a retention strategy (a customer
who just made their first purchase needs different treatment than one who
hasn't bought in over a year, even though both currently look "low value").

We choose **K=4**: it trades a modest amount of silhouette score for segments
that map onto distinct, actionable business situations. This is a judgment call,
not a purely statistical one — worth being explicit about that.

## Assigning business labels — only now, based on the actual numbers above

(Cluster numbers depend on KMeans' random initialization ordering — read the
table above, don't assume cluster 0 is always the same group.)

This labeling logic is derived from the cluster statistics, not assumed —
if you re-run this notebook and KMeans assigns different cluster numbers
(which can happen with different random states), the labels still land on
the right groups because we're sorting by the actual Recency/Monetary values,
not hardcoding cluster index 0/1/2/3 to fixed names.

## Save

In [1]:
import pandas as pd
pd.set_option('future.infer_string', False)
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

customer_features = pd.read_csv("../data/processed/customer_features.csv")

features = customer_features[["Recency", "Frequency_log", "Monetary_log"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

k_range = range(2, 9)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
for k, inertia, sil in zip(k_range, inertias, silhouettes):
    print(f"k={k}  inertia={inertia:.1f}  silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o"); axes[0].set_title("Elbow Method")
axes[1].plot(list(k_range), silhouettes, marker="o", color="orange"); axes[1].set_title("Silhouette Score")
plt.tight_layout(); plt.savefig("../reports/figures/kmeans_k_selection.png"); plt.close()

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_features["Cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = customer_features.groupby("Cluster")[
    ["Recency", "Frequency", "Monetary", "AvgOrderValue", "UniqueProducts"]
].mean().round(1)
print(cluster_summary)

# FIX: Recency alone doesn't reliably separate New vs Regular here (99.9 vs 102.9 days
# is noise). Frequency does (1.8 vs 5.9 orders) -- use it for that split instead.
label_map = {}
sorted_by_recency_desc = cluster_summary.sort_values("Recency", ascending=False)
lapsed_cluster = sorted_by_recency_desc.index[0]

remaining = cluster_summary.drop(lapsed_cluster)
best_cluster = remaining.sort_values("Monetary", ascending=False).index[0]

remaining2 = remaining.drop(best_cluster)
new_cluster = remaining2.sort_values("Frequency", ascending=True).index[0]   # <-- fixed
regular_cluster = remaining2.drop(new_cluster).index[0]

label_map[lapsed_cluster] = "Lapsed"
label_map[best_cluster] = "Best Customers"
label_map[new_cluster] = "New / Low Engagement"
label_map[regular_cluster] = "Regular"

customer_features["Segment"] = customer_features["Cluster"].map(label_map)
print(customer_features["Segment"].value_counts())
print(customer_features.groupby("Segment")[["Recency","Frequency","Monetary"]].mean().round(1))

fig, ax = plt.subplots(figsize=(8, 5))
customer_features["Segment"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Customer Segment Sizes")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("../reports/figures/segment_sizes.png"); plt.close()

customer_features.to_csv("../data/processed/customer_features_segmented.csv", index=False)
print("saved:", customer_features.shape)


k=2  inertia=8880.7  silhouette=0.4187
k=3  inertia=5716.0  silhouette=0.4010
k=4  inertia=4473.0  silhouette=0.3616
k=5  inertia=3627.8  silhouette=0.3664
k=6  inertia=3072.5  silhouette=0.3469
k=7  inertia=2726.3  silhouette=0.3353
k=8  inertia=2464.2  silhouette=0.3155
         Recency  Frequency  Monetary  AvgOrderValue  UniqueProducts
Cluster                                                             
0          102.9        1.8     460.7           26.3            32.0
1          492.1        1.7     521.3           45.0            29.9
2           38.4       23.4   13656.7          106.6           234.0
3           99.9        5.9    2075.8           41.4            96.6
Segment
Regular                 1862
Lapsed                  1646
New / Low Engagement    1494
Best Customers           876
Name: count, dtype: int64
                      Recency  Frequency  Monetary
Segment                                           
Best Customers           38.4       23.4   13656.7
Lapsed    